In [1]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [3]:
model = ChatGoogleGenerativeAI(model='gemini-2.5-flash')

In [5]:
class BlogState(TypedDict):
    title:str
    outline:str
    content:str

In [6]:
def create_outline(state: BlogState)-> BlogState:
    # fetch title
    title = state['title']

    # call llm
    prompt =f"Generate an outline for the blog on a topic - {title}"
    outline =model.invoke(prompt).content
    # update state
    state['outline'] = outline

    return state

In [7]:
def create_blog(state: BlogState)-> BlogState:
    # fetch title
    title = state['title']
    outline = state['outline']

    # call llm
    prompt =f"Write a short blog on the title - {title} using the following outline \n {outline}"
    content =model.invoke(prompt).content
    # update state
    state['content'] = content

    return state

In [9]:
graph = StateGraph(BlogState)

#nodes
graph.add_node('create_outline', create_outline)
graph.add_node('create_blog', create_blog)

#edge
graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', END)

workflow = graph.compile()

In [11]:
inital_state = {'title': 'Rise of LangGraph'}

final_state = workflow.invoke(inital_state)

print(final_state)

{'title': 'Rise of LangGraph', 'outline': 'Okay, here\'s a detailed outline for a blog post on "The Rise of LangGraph," aiming to inform, excite, and guide readers.\n\n---\n\n## Blog Title Options:\n\n*   The Rise of LangGraph: Building Robust, Stateful LLM Agents\n*   Beyond Chains: How LangGraph is Revolutionizing LLM Application Development\n*   LangGraph Unleashed: Crafting Intelligent, Autonomous AI Workflows\n*   Why LangGraph? The Future of Stateful LLM Orchestration\n\n## Meta Description:\n\nExplore the rise of LangGraph, a powerful library for building stateful, multi-actor LLM applications. Learn how it solves the limitations of traditional LLM chains, enabling robust, self-correcting AI agents and complex workflows.\n\n---\n\n## Blog Outline: The Rise of LangGraph\n\n### I. Introduction: The Evolution of LLM Applications\n\n*   **A. The Promise of LLMs:** Briefly touch on the excitement around Large Language Models and their potential.\n*   **B. The Challenge of Complexity: